2.1
1.输出特征图尺寸：16 × 16 × 16（通道数 × 高 × 宽）
2.单个输出通道的一个像素值需要进行的乘法次数：3*5*5=75 次

In [1]:
import numpy as np

def max_pool2d_manual(x, kernel_size, stride=1, padding=0):
    # x: (batch, channels, height, width)
    # kernel_size: int or (int, int)
    # stride: int or (int, int)
    # padding: int or (int, int)
    if isinstance(kernel_size, int):
        k_h = k_w = kernel_size
    else:
        k_h, k_w = kernel_size
    if isinstance(stride, int):
        s_h = s_w = stride
    else:
        s_h, s_w = stride
    if isinstance(padding, int):
        p_h = p_w = padding
    else:
        p_h, p_w = padding

    b, c, h, w = x.shape
    h_out = (h + 2 * p_h - k_h) // s_h + 1
    w_out = (w + 2 * p_w - k_w) // s_w + 1

    # 填充
    x_pad = np.pad(x, ((0,0), (0,0), (p_h, p_h), (p_w, p_w)), mode='constant')
    output = np.zeros((b, c, h_out, w_out))

    for i in range(h_out):
        for j in range(w_out):
            h_start = i * s_h
            h_end = h_start + k_h
            w_start = j * s_w
            w_end = w_start + k_w
            window = x_pad[:, :, h_start:h_end, w_start:w_end]
            output[:, :, i, j] = np.max(window, axis=(2,3))
    return output

3.1
1.一个 5×5 卷积层（不带偏置）的参数量：25 × C²
2.两个串联的 3×3 卷积层（不带偏置，通道数均为 C）的总参数量：18 × C²

In [2]:
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride, padding):
        super(NiNBlock, self).__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU(),
            nn.Conv2d(out_channels, out_channels, kernel_size=1),
            nn.ReLU()
        )

    def forward(self, x):
        return self.block(x)

4..1
经过 Batch Normalization 转化后的最终输出值：
y1 = -0.683
y2 = 0.106
y3 = 1.894
y4 = 3.683

In [4]:
import torch
import torch.nn as nn

class Residual(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(Residual, self).__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, stride=stride)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU()

        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride)
        else:
            self.conv3 = None

    def forward(self, x):
        y = self.relu(self.bn1(self.conv1(x)))
        y = self.bn2(self.conv2(y))
        if self.conv3:
            x = self.conv3(x)
        y += x
        return self.relu(y)

5.1
1.底层特征提取层已在源数据集上学到通用特征（如边缘、纹理），用小学习率避免破坏；顶层输出层需要适应新类别，用大学习率快速学习。
2.如果目标数据集非常小且与源数据集非常相似，应采取以下微调策略：冻结大部分底层卷积层，只微调最后几层或全连接层；使用较小学习率；增加数据增广、Dropout、早停等正则化手段防止过拟合。

In [5]:
import torchvision.transforms as transforms

transform_pipeline = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
    transforms.ToTensor()
])

6.1
IoU = 1/7 ≈ 0.142857

In [7]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, labels, epsilon=0.1, num_classes=None):
    if num_classes is None:
        num_classes = logits.shape[-1]
    
    batch_size = logits.shape[0]
    log_probs = F.log_softmax(logits, dim=-1)
    
    # 构建平滑标签
    smooth_labels = torch.full_like(log_probs, epsilon / (num_classes - 1))
    smooth_labels.scatter_(1, labels.unsqueeze(1), 1 - epsilon)
    
    loss = - (smooth_labels * log_probs).sum(dim=-1).mean()
    return loss